<a href="https://colab.research.google.com/github/Skydan111/CubeSat-Security-Simulator/blob/main/ai/Warehouse_Monitor_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/86

In [1]:
from unsloth import FastLanguageModel
print("Unsloth OK")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth OK


In [2]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-3b-instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)
print("Model loaded")

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded


In [3]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

FastLanguageModel.for_inference(model)

messages = [{
    "role": "user",
    "content": "Analysiere die Telemetriedaten und erstelle einen Bericht für den Lageroperator.\n\nParameter: Temperatur\nSchwellenwert: min 2,0°C / max 8,0°C\nAktueller Wert: 10,3°C\nAnalysezeitraum: letzte 30 Minuten (30 Messpunkte)\nDurchschnitt: 9,1°C\nMinimum: 7,8°C\nMaximum: 10,3°C\nÜberschreitung: +2,3°C, Andauer: 14 Minuten\nBeginn der Anomalie: 10:22:15"
}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== Basismodell (vor dem Fine-Tuning) ===")
print(response)



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

=== Basismodell (vor dem Fine-Tuning) ===
**Lagerbericht: Temperaturüberwachung**

**Zusammenfassung:**

In den letzten 30 Minuten haben wir eine Temperatur von 10,3°C gemessen. Dies liegt über dem Schwellenwert von 8,0°C. Der Durchschnitt der letzten 30 Minuten beträgt 9,1°C, was ebenfalls über dem Schwellenwert liegt.

**Überschreitung:**

Wir haben eine Temperaturüberwachung von +2,3°C, was bedeutet, dass die Temperatur 2,3°C über dem Schwellenwert liegt. Diese Überwachung hat 14 Minuten gedauert.

**Beginn der Anomalie:**

Die Anomalie begann um 10:22:15.

**Empfehlungen:**

* Überprüfen Sie die Klimaanlage und die Umgebungstemperatur, um sicherzustellen, dass sie ordnungsgemäß funktioniert.
* Überprüfen Sie die Luftzirkulation und die Ventilationskanäle, um sicherzustellen, dass sie ordnungsgemäß funktionieren.
* Überprüfen Sie die Temperaturregelung, um sicherzustellen, dass sie ordnungsgemäß funktioniert.

**Schlussfolgerung:**

Wir empfehlen, die Anomalie weiterhin zu überwache

In [4]:
# LoRA-Konfiguration – nur kleine Gewichtsanpassungen trainieren
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA-Konfiguration bereit")


Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA-Konfiguration bereit


In [5]:
from datasets import load_dataset

# Datensatz laden
dataset = load_dataset("json", data_files="/content/dataset.jsonl", split="train")

# Prompt-Format definieren
def format_prompt(example):
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    }

dataset = dataset.map(format_prompt)
print(f"Datensatz geladen: {len(dataset)} Beispiele")
print(dataset[0]["text"][:300])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Datensatz geladen: 35 Beispiele
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Analysiere die Telemetriedaten und erstelle einen Bericht für den Lageroperator.

Parameter: Temperatur
Schwellenwert: min 2,0°C / max 8,0°C
Aktueller Wert: 10,3°C
Analysezeitraum: letzte 30 Minuten (30 Messpunkte)
Durchschnitt: 9,1°C
Minimu


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training starten
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 5,
        output_dir = "/content/warehouse_model",
        save_strategy = "no",
        report_to = "none",
    ),
)

print("Training wird gestartet...")
trainer.train()
print("Training abgeschlossen!")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/35 [00:00<?, ? examples/s]

Training wird gestartet...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 35 | Num Epochs = 3 | Total steps = 15
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.645827
10,1.200853
15,0.956352


Training abgeschlossen!


In [9]:
# Test nach dem Fine-Tuning – dasselbe Beispiel wie vorher
FastLanguageModel.for_inference(model)

messages = [{
    "role": "user",
    "content": "Analysiere die Telemetriedaten und erstelle einen Bericht für den Lageroperator.\n\nParameter: Temperatur\nSchwellenwert: min 2,0°C / max 8,0°C\nAktueller Wert: 10,3°C\nAnalysezeitraum: letzte 30 Minuten (30 Messpunkte)\nDurchschnitt: 9,1°C\nMinimum: 7,8°C\nMaximum: 10,3°C\nÜberschreitung: +2,3°C, Andauer: 14 Minuten\nBeginn der Anomalie: 10:22:15"
}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== Trainiertes Modell (nach dem Fine-Tuning) ===")
print(response)


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


=== Trainiertes Modell (nach dem Fine-Tuning) ===
ANOMALIE ERKANNT – Temperaturüberschreitung

Seit 10:22:15 befindet sich die Temperatur außerhalb des definierten Sollbereichs. Der aktuelle Messwert beträgt 10,3°C und liegt damit 2,3°C über dem zulässigen Maximum von 8,0°C. Die Andauer der Überschreitung beträgt derzeit 14 Minuten.

Im Analysezeitraum der letzten 30 Minuten wurde ein Durchschnittswert von 9,1°C ermittelt. Die Temperaturwerte lagen überwiegend im Sollbereich, mit einer höchsten Messwerte von 10,3°C. Der aktuelle Messwert befindet sich damit auf einem Höchstwert von 10,3°C und ist damit 2,3°C oberhalb des zulässigen Grenzwerts.

Die vorliegenden Messdaten deuten auf eine anhaltende Temperaturüberschreitung hin. Da die Andauer der Überschreitung länger als 10 Minuten beträgt, empfängt der Betriebsoperator eine erhöhte Sorgfalt. Zusätzlich ist die länger anhaltende Überschreitung mit einem erhöhten Risiko für mögliche Temperaturabweichungen, Schäden an Lagerbeständen oder

In [8]:
# Erneutes Training mit mehr Epochen
trainer2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 6,
        learning_rate = 1e-4,
        fp16 = True,
        logging_steps = 5,
        output_dir = "/content/warehouse_model",
        save_strategy = "no",
        report_to = "none",
    ),
)

print("Erneutes Training wird gestartet...")
trainer2.train()
print("Training abgeschlossen!")


Erneutes Training wird gestartet...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 35 | Num Epochs = 6 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
5,0.779756
10,0.582646
15,0.464920
20,0.410380
25,0.381940
30,0.359280


Training abgeschlossen!


In [10]:
# Modell speichern
model.save_pretrained("/content/warehouse_finetuned")
tokenizer.save_pretrained("/content/warehouse_finetuned")
print("Modell gespeichert unter /content/warehouse_finetuned")


Unsloth: Restored added_tokens_decoder metadata in /content/warehouse_finetuned/tokenizer_config.json.


Modell gespeichert unter /content/warehouse_finetuned


In [11]:
# Auf Google Drive speichern
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copytree(
    "/content/warehouse_finetuned",
    "/content/drive/MyDrive/warehouse_finetuned"
)
print("Modell auf Google Drive gespeichert")



Mounted at /content/drive
Modell auf Google Drive gespeichert
